In [1]:
import os
import re
import glob
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter
import matplotlib.pyplot as plt


# ----------------------------
# Config dos grupos desejados
# ----------------------------
KEYWORDS = [
    "DefaultReward",
    "RewardHard",
    "RewardLowerQuantile",
    "RewardMaxMedian",
]

VARIANT_ORDER = [
    "DefaultReward",
    "RewardHard",
    "RewardLowerQuantile",
    "RewardMaxMedian",
]

def format_x_1e5(x, pos):
    if x == 0:
        return "0"
    return f"{int(x/1e5)}×10⁵"

# ----------------------------
# Tree printer
# ----------------------------
def print_tree_dir(path, prefix=""):
    items = sorted(os.listdir(path))
    for i, item in enumerate(items):
        full = os.path.join(path, item)
        connector = "├── " if i < len(items) - 1 else "└── "
        print(prefix + connector + item)
        if os.path.isdir(full):
            new_prefix = prefix + ("│   " if i < len(items) - 1 else "    ")
            print_tree_dir(full, new_prefix)


# ----------------------------
# Discovery helpers
# ----------------------------
def find_experiments(root):
    """Experiment dirs containing progress.csv (walk recursively)."""
    exps = []
    for r, _, files in os.walk(root):
        if "progress.csv" in files:
            exps.append(r)
    return sorted(exps)


def extract_group_and_variant(name):
    """
    Examples
    --------
    'COMA-DefaultReward (0.5,0.7,0.5,0.6)'
        -> algo='COMA', variant='DefaultReward', group='COMA-DefaultReward'

    'MATRPO-RewardMaxMedian (0.5,0.7,0.5,0.6) - 3'
        -> algo='MATRPO', variant='RewardMaxMedian', group='MATRPO-RewardMaxMedian'

    Retorna (None, None, None) se não for um dos grupos desejados.
    """
    base = name.split(" - ")[0] if " - " in name else name

    if "-" not in base:
        return None, None, None

    algo = base.split("-", 1)[0].strip()

    variant = None
    for k in KEYWORDS:
        if k in base:
            variant = k
            break

    if variant is None:
        return None, None, None

    group = f"{algo}-{variant}"
    return algo, variant, group


def is_target_experiment(name):
    algo, variant, group = extract_group_and_variant(name)
    return group is not None


# ----------------------------
# Reward + AUC from progress.csv
# ----------------------------
def compute_reward_and_auc(root, reward_col="episode_reward_mean"):
    reward_rows, auc_rows = [], []

    for exp in find_experiments(root):
        seed = os.path.basename(exp)

        algo, variant, group = extract_group_and_variant(seed)
        if group is None:
            continue

        csv = os.path.join(exp, "progress.csv")
        try:
            df = pd.read_csv(csv)
        except Exception:
            continue

        if reward_col not in df.columns:
            continue

        rewards = pd.to_numeric(df[reward_col], errors="coerce").dropna()
        if rewards.empty:
            continue

        # ----------------------------
        # FINAL REWARD (igual antes)
        # ----------------------------
        reward_rows.append({
            "algo": algo,
            "variant": variant,
            "group": group,
            "seed": seed,
            "mean_reward": float(rewards.mean())
        })

        # ============================================================
        # 🔥 AUC NORMALIZADO (ESTILO PAPER - TABELA 3)
        # ============================================================

        r = rewards.values.astype(float)

        r_min = r.min()
        r_max = r.max()

        # evita divisão por zero
        if (r_max - r_min) < 1e-8:
            continue

        # 1. normalização 0–1  <<< ALTERADO
        r_norm = (r - r_min) / (r_max - r_min)

        # 2. eixo do tempo (índice) <<< ALTERADO
        x = np.arange(len(r_norm))

        # 3. AUC normalizado <<< ALTERADO
        auc = np.trapz(r_norm, x) / (x[-1] - x[0])

        auc_rows.append({
            "algo": algo,
            "variant": variant,
            "group": group,
            "seed": seed,
            "auc": float(auc)
        })

    # ----------------------------
    # (RESTO DA FUNÇÃO NÃO MUDA)
    # ----------------------------

    reward_df = pd.DataFrame(reward_rows)
    auc_df = pd.DataFrame(auc_rows)

    if reward_df.empty:
        reward_grp = pd.DataFrame(columns=["algo", "variant", "mean", "std", "count", "mean_std"])
    else:
        reward_grp = (
            reward_df
            .groupby(["algo", "variant"])["mean_reward"]
            .agg(["mean", "std", "count"])
            .reset_index()
        )
        reward_grp["variant"] = pd.Categorical(
            reward_grp["variant"],
            categories=VARIANT_ORDER,
            ordered=True
        )
        reward_grp = reward_grp.sort_values(["algo", "variant"]).reset_index(drop=True)
        reward_grp["mean_std"] = (
            reward_grp["mean"].round(3).astype(str)
            + " ± "
            + reward_grp["std"].round(3).astype(str)
        )

    if auc_df.empty:
        auc_grp = pd.DataFrame(columns=["algo", "variant", "mean", "std", "count", "mean_std"])
    else:
        auc_grp = (
            auc_df
            .groupby(["algo", "variant"])["auc"]
            .agg(["mean", "std", "count"])
            .reset_index()
        )
        auc_grp["variant"] = pd.Categorical(
            auc_grp["variant"],
            categories=VARIANT_ORDER,
            ordered=True
        )
        auc_grp = auc_grp.sort_values(["algo", "variant"]).reset_index(drop=True)
        auc_grp["mean_std"] = (
            auc_grp["mean"].round(3).astype(str)
            + " ± "
            + auc_grp["std"].round(3).astype(str)
        )

    return reward_grp, auc_grp, reward_df, auc_df


# ----------------------------
# TensorBoard tag utilities
# ----------------------------
RAY_PREFIX = re.compile(r"^(ray/(tune|train|rllib)/)")


def strip_ray(tag):
    return RAY_PREFIX.sub("", tag)


REWARD_PREF = [
    "episode_reward_mean",
]

REWARD_REGEX = re.compile(r"(episode|ep).*(reward|return).*(mean|avg)", re.IGNORECASE)

X_PREF = [
    "timesteps_total",
    "num_env_steps_sampled",
    "num_agent_steps_sampled",
    "training_iteration",
]


def choose_reward_tag(tags):
    for t in REWARD_PREF:
        if t in tags:
            return t
    for t in tags:
        if REWARD_REGEX.search(t):
            return t
    return None


def choose_x_tag(tags):
    for t in X_PREF:
        if t in tags:
            return t
    return None


def get_event_files(d):
    return sorted(glob.glob(os.path.join(d, "**", "events.out.tfevents.*"), recursive=True))


# ----------------------------
# Smoothing + plotting
# ----------------------------
def _smooth(arr, sigma):
    if sigma is None or sigma <= 0:
        return arr
    try:
        from scipy.ndimage import gaussian_filter1d
        return gaussian_filter1d(arr, sigma=sigma)
    except Exception:
        w = int(max(3, round(2 * sigma + 1)))
        if w % 2 == 0:
            w += 1
        kernel = np.ones(w) / w
        return np.convolve(arr, kernel, mode="same")


def _safe_minmax(values):
    """
    GUARANTEE:
      - vmin = min(values)
      - vmax = max(values)
      - if degenerate, widen slightly
    """
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return 0.0, 1.0

    vmin = float(np.min(values))
    vmax = float(np.max(values))

    if not np.isfinite(vmin) or not np.isfinite(vmax) or (vmax - vmin) < 1e-12:
        vmin -= 1.0
        vmax += 1.0

    return vmin, vmax


def _norm01(y, vmin, vmax):
    denom = (vmax - vmin) if (vmax - vmin) != 0 else 1.0
    return np.clip((y - vmin) / denom, 0.0, 1.0)


def plot_raw(xs, m, s, title, save, smooth_sigma):
    import matplotlib.pyplot as plt

    m2, s2 = _smooth(m, smooth_sigma), _smooth(s, smooth_sigma)

    plt.figure(figsize=(20, 10))
    ax = plt.gca()

    plt.plot(xs, m2, label="episode_reward_mean")
    plt.fill_between(xs, m2 - s2, m2 + s2, alpha=0.2)

    ax.set_xlabel("Step", fontsize=30)
    ax.set_ylabel("Reward", fontsize=30)
    ax.tick_params(axis='both', labelsize=26)

    plt.legend(fontsize=24)
    plt.grid(True)

    os.makedirs(os.path.dirname(save), exist_ok=True)
    plt.savefig(save, dpi=300, bbox_inches="tight")
    plt.close()


def plot_norm(xs, m, s, title, save, smooth_sigma, vmin, vmax):
    import matplotlib.pyplot as plt

    m2, s2 = _smooth(m, smooth_sigma), _smooth(s, smooth_sigma)

    nm = _norm01(m2, vmin, vmax)
    denom = (vmax - vmin) if (vmax - vmin) != 0 else 1.0
    ns = s2 / denom
    lower = np.clip(nm - ns, 0, 1)
    upper = np.clip(nm + ns, 0, 1)

    plt.figure(figsize=(20, 10))
    ax = plt.gca()

    max_x = xs.max()
    ticks = np.arange(0, max_x + 1e5, 1e5)
    ax.set_xticks(ticks)
    ax.xaxis.set_major_formatter(FuncFormatter(format_x_1e5))

    plt.plot(xs, nm, label="episode_reward_mean")
    plt.fill_between(xs, lower, upper, alpha=0.2)
    plt.ylim(0, 1)

    ax.set_xlabel("Step", fontsize=30)
    ax.set_ylabel("Reward", fontsize=30)
    ax.tick_params(axis='both', labelsize=26)

    plt.legend(fontsize=26)
    plt.grid(True)

    os.makedirs(os.path.dirname(save), exist_ok=True)
    plt.savefig(save, dpi=150, bbox_inches="tight")
    plt.close()

# ----------------------------
# Charts: agrupados por algoritmo + keyword
# ----------------------------
def make_charts(
    root,
    results_dir,
    smooth_sigma=3.0,
    norm_scope="global",  # "global" or "per_group"
    max_x=None,
):
    """
    Guarantees:
      - Normalized plot uses MIN-MAX
      - 0 == lowest value, 1 == highest value
      - scope controls whether min/max is computed globally or per group

    Grouping:
      - one chart group per "{algo}-{variant}"
      - examples:
          COMA-DefaultReward
          COMA-RewardHard
          MATRPO-RewardMaxMedian

    X axis:
      - uses x_tag if found (timesteps_total, training_iteration, ...)
      - otherwise uses event.step
    """
    try:
        from tensorflow.python.summary.summary_iterator import summary_iterator
    except Exception as e:
        print(f"[WARN] TensorFlow not available — skipping charts ({e})")
        return

    groups = defaultdict(list)

    for exp in find_experiments(root):
        seed = os.path.basename(exp)

        algo, variant, group = extract_group_and_variant(seed)
        if group is None:
            continue

        groups[group].append(exp)

    if not groups:
        print("[WARN] No matching experiment folders found (with progress.csv).")
        return

    def extract_group_series(seed_dirs):
        event_files = []
        for sd in seed_dirs:
            event_files.extend(get_event_files(sd))
        if not event_files:
            return None

        tag_counts = Counter()
        for ef in event_files:
            try:
                for e in summary_iterator(ef):
                    if not hasattr(e, "summary") or e.summary is None:
                        continue
                    for v in e.summary.value:
                        tag_counts[strip_ray(v.tag)] += 1
            except Exception:
                pass

        y_tag = choose_reward_tag(tag_counts.keys())
        if not y_tag:
            return None

        x_tag = choose_x_tag(tag_counts.keys())

        y_by_step = defaultdict(list)
        x_by_step = defaultdict(list) if x_tag else None

        for ef in event_files:
            try:
                for e in summary_iterator(ef):
                    if not hasattr(e, "summary") or e.summary is None:
                        continue
                    step = int(e.step)
                    for v in e.summary.value:
                        tag = strip_ray(v.tag)
                        if x_tag and tag == x_tag:
                            x_by_step[step].append(float(v.simple_value))
                        if tag == y_tag:
                            y_by_step[step].append(float(v.simple_value))
            except Exception:
                pass

        if not y_by_step:
            return None

        steps = sorted(y_by_step.keys())

        if x_tag:
            xs = np.array(
                [
                    np.mean(x_by_step[s]) if s in x_by_step and len(x_by_step[s]) > 0 else float(s)
                    for s in steps
                ],
                dtype=float
            )
        else:
            xs = np.array([float(s) for s in steps], dtype=float)

        m = np.array([np.mean(y_by_step[s]) for s in steps], dtype=float)
        s = np.array([np.std(y_by_step[s]) for s in steps], dtype=float)

        order = np.argsort(xs)
        xs = xs[order]
        m = m[order]
        s = s[order]

        if max_x is not None:
            mask = xs <= max_x
            xs = xs[mask]
            m = m[mask]
            s = s[mask]

        if len(xs) == 0:
            return None

        return xs, m, s, y_tag, x_tag

    cached = {}
    global_smoothed_means = []

    for group, seed_dirs in sorted(groups.items()):
        triple = extract_group_series(seed_dirs)
        if triple is None:
            continue

        xs, m, s, y_tag, x_tag = triple
        m_sm = _smooth(m, smooth_sigma)

        cached[group] = (xs, m, s, m_sm, y_tag, x_tag)
        global_smoothed_means.extend(m_sm.tolist())

    if not cached:
        print("[WARN] No chartable data found (no tfevents or no reward tag).")
        return

    global_vmin, global_vmax = _safe_minmax(np.array(global_smoothed_means))

    print("[Charts] X auto (x_tag if exists else step).")
    print(f"[Charts] Normalization = MIN-MAX with scope={norm_scope}.")
    print(f"[Charts] Global min-max (on smoothed means): vmin={global_vmin:.6f}, vmax={global_vmax:.6f}")

    charts_root = os.path.join(results_dir, "charts")

    for group, (xs, m, s, m_sm, y_tag, x_tag) in cached.items():
        out_dir = os.path.join(charts_root, group)

        if norm_scope == "per_group":
            vmin, vmax = _safe_minmax(m_sm)
        else:
            vmin, vmax = global_vmin, global_vmax

        plot_raw(
            xs, m, s,
            title=f"{group} — raw (x={x_tag or 'step'}, y={y_tag})",
            save=os.path.join(out_dir, "raw.png"),
            smooth_sigma=smooth_sigma
        )

        plot_norm(
            xs, m, s,
            title=f"{group} — normalized (x={x_tag or 'step'}, y={y_tag})",
            save=os.path.join(out_dir, "normalized.png"),
            smooth_sigma=smooth_sigma,
            vmin=vmin,
            vmax=vmax
        )

        print(
            f"✔ charts: {group} | x={x_tag or 'step'} | y={y_tag} "
            f"| norm vmin={vmin:.6f} vmax={vmax:.6f}"
        )


# ----------------------------
# MAIN ENTRY (Notebook-safe)
# ----------------------------
def run_all(
    root_folder,
    results_folder_name=None,
    reward_col="episode_reward_mean",
    print_tree=True,
    do_charts=True,
    smooth_sigma=3.0,
    norm_scope="global",  # "global" or "per_group"
    show_seed_tables=False,
    max_x=None,
):
    root_folder = os.path.abspath(root_folder)
    if not os.path.isdir(root_folder):
        raise ValueError(f"Folder not found: {root_folder}")

    base = os.path.basename(os.path.normpath(root_folder))
    if results_folder_name is None:
        results_folder_name = f"{base}_results"

    results_dir = os.path.join(os.path.dirname(root_folder), results_folder_name)
    os.makedirs(results_dir, exist_ok=True)

    if print_tree:
        print("📂 Folder tree:")
        print_tree_dir(root_folder)

    reward_grp, auc_grp, reward_seeds, auc_seeds = compute_reward_and_auc(root_folder, reward_col)

    reward_csv = os.path.join(results_dir, "reward_stats_grouped.csv")
    auc_csv = os.path.join(results_dir, "auc_grouped.csv")
    reward_seed_csv = os.path.join(results_dir, "reward_stats_per_seed.csv")
    auc_seed_csv = os.path.join(results_dir, "auc_per_seed.csv")

    reward_grp.to_csv(reward_csv, index=False)
    auc_grp.to_csv(auc_csv, index=False)
    reward_seeds.to_csv(reward_seed_csv, index=False)
    auc_seeds.to_csv(auc_seed_csv, index=False)

    print("\n✅ Saved:")
    print(reward_csv)
    print(auc_csv)
    print(reward_seed_csv)
    print(auc_seed_csv)

    try:
        from IPython.display import display

        display(reward_grp)
        display(auc_grp)

        if show_seed_tables:
            if not reward_seeds.empty:
                reward_seeds = reward_seeds.sort_values(["algo", "variant", "seed"])
            if not auc_seeds.empty:
                auc_seeds = auc_seeds.sort_values(["algo", "variant", "seed"])

            display(reward_seeds)
            display(auc_seeds)

    except Exception:
        print("\n[Reward grouped]\n", reward_grp)
        print("\n[AUC grouped]\n", auc_grp)

        if show_seed_tables:
            print("\n[Reward per seed]\n", reward_seeds.sort_values(["algo", "variant", "seed"]))
            print("\n[AUC per seed]\n", auc_seeds.sort_values(["algo", "variant", "seed"]))

    if do_charts:
        make_charts(
            root=root_folder,
            results_dir=results_dir,
            smooth_sigma=smooth_sigma,
            norm_scope=norm_scope,
            max_x=max_x
        )
        print("\n📊 Charts in:", os.path.join(results_dir, "charts"))

    print("\n[DONE] Results folder:", results_dir)
    return results_dir

In [9]:
run_all(
    root_folder="/mnt/ssd1/wesley/BusEnv/exp_results",
    results_folder_name="runs_abril_results",
    reward_col="episode_reward_mean",
    print_tree=False,
    do_charts=True,
    smooth_sigma=3.0,
    norm_scope="global",
    show_seed_tables=True,
    max_x=None
)



✅ Saved:
/mnt/ssd1/wesley/BusEnv/runs_abril_results/reward_stats_grouped.csv
/mnt/ssd1/wesley/BusEnv/runs_abril_results/auc_grouped.csv
/mnt/ssd1/wesley/BusEnv/runs_abril_results/reward_stats_per_seed.csv
/mnt/ssd1/wesley/BusEnv/runs_abril_results/auc_per_seed.csv


,algo,variant,mean,std,count,mean_std
0,COMA,DefaultReward,1048.761336,6.871521,25,1048.761 ± 6.872
1,COMA,RewardHard,-219.223015,2.586556,25,-219.223 ± 2.587
2,COMA,RewardLowerQuantile,6.733299,0.009687,25,6.733 ± 0.01
3,COMA,RewardMaxMedian,1529.814084,5.997720,25,1529.814 ± 5.998
4,HAPPO,DefaultReward,358.297950,0.048621,25,358.298 ± 0.049
5,HAPPO,RewardHard,-529.614583,0.050809,25,-529.615 ± 0.051
6,HAPPO,RewardLowerQuantile,5.895350,0.062814,25,5.895 ± 0.063
7,HAPPO,RewardMaxMedian,498.205149,5.473009,25,498.205 ± 5.473
8,HATRPO,DefaultReward,1366.323384,10.451410,25,1366.323 ± 10.451
9,HATRPO,RewardHard,-32.212958,0.552035,25,-32.213 ± 0.552


,algo,variant,mean,std,count,mean_std
0,COMA,DefaultReward,0.583804,0.005252,25,0.584 ± 0.005
1,COMA,RewardHard,0.610033,0.004740,25,0.61 ± 0.005
2,COMA,RewardLowerQuantile,0.683391,0.020942,25,0.683 ± 0.021
3,COMA,RewardMaxMedian,0.585423,0.003281,25,0.585 ± 0.003
4,HAPPO,DefaultReward,0.874581,0.003241,25,0.875 ± 0.003
5,HAPPO,RewardHard,0.206742,0.000369,25,0.207 ± 0.0
6,HAPPO,RewardLowerQuantile,0.692052,0.042169,25,0.692 ± 0.042
7,HAPPO,RewardMaxMedian,0.869313,0.007530,25,0.869 ± 0.008
8,HATRPO,DefaultReward,0.822548,0.006856,25,0.823 ± 0.007
9,HATRPO,RewardHard,0.932350,0.001198,25,0.932 ± 0.001


,algo,variant,group,seed,mean_reward
0,COMA,DefaultReward,COMA-DefaultReward,"COMA-DefaultReward (0.5,0.7,0.5,0.6) - 1",1061.519502
1,COMA,DefaultReward,COMA-DefaultReward,"COMA-DefaultReward (0.5,0.7,0.5,0.6) - 10",1057.617931
2,COMA,DefaultReward,COMA-DefaultReward,"COMA-DefaultReward (0.5,0.7,0.5,0.6) - 11",1045.600055
3,COMA,DefaultReward,COMA-DefaultReward,"COMA-DefaultReward (0.5,0.7,0.5,0.6) - 12",1043.468634
4,COMA,DefaultReward,COMA-DefaultReward,"COMA-DefaultReward (0.5,0.7,0.5,0.6) - 13",1049.626774
...,...,...,...,...,...
895,MATRPO,RewardMaxMedian,MATRPO-RewardMaxMedian,"MATRPO-RewardMaxMedian (0.5,0.7,0.5,0.6) - 5",2089.011197
896,MATRPO,RewardMaxMedian,MATRPO-RewardMaxMedian,"MATRPO-RewardMaxMedian (0.5,0.7,0.5,0.6) - 6",2089.011197
897,MATRPO,RewardMaxMedian,MATRPO-RewardMaxMedian,"MATRPO-RewardMaxMedian (0.5,0.7,0.5,0.6) - 7",2089.011197
898,MATRPO,RewardMaxMedian,MATRPO-RewardMaxMedian,"MATRPO-RewardMaxMedian (0.5,0.7,0.5,0.6) - 8",2089.011197


,algo,variant,group,seed,auc
0,COMA,DefaultReward,COMA-DefaultReward,"COMA-DefaultReward (0.5,0.7,0.5,0.6) - 1",0.593393
1,COMA,DefaultReward,COMA-DefaultReward,"COMA-DefaultReward (0.5,0.7,0.5,0.6) - 10",0.590323
2,COMA,DefaultReward,COMA-DefaultReward,"COMA-DefaultReward (0.5,0.7,0.5,0.6) - 11",0.581510
3,COMA,DefaultReward,COMA-DefaultReward,"COMA-DefaultReward (0.5,0.7,0.5,0.6) - 12",0.579914
4,COMA,DefaultReward,COMA-DefaultReward,"COMA-DefaultReward (0.5,0.7,0.5,0.6) - 13",0.584567
...,...,...,...,...,...
895,MATRPO,RewardMaxMedian,MATRPO-RewardMaxMedian,"MATRPO-RewardMaxMedian (0.5,0.7,0.5,0.6) - 5",0.856666
896,MATRPO,RewardMaxMedian,MATRPO-RewardMaxMedian,"MATRPO-RewardMaxMedian (0.5,0.7,0.5,0.6) - 6",0.856666
897,MATRPO,RewardMaxMedian,MATRPO-RewardMaxMedian,"MATRPO-RewardMaxMedian (0.5,0.7,0.5,0.6) - 7",0.856666
898,MATRPO,RewardMaxMedian,MATRPO-RewardMaxMedian,"MATRPO-RewardMaxMedian (0.5,0.7,0.5,0.6) - 8",0.856666


[Charts] X auto (x_tag if exists else step).
[Charts] Normalization = MIN-MAX with scope=global.
[Charts] Global min-max (on smoothed means): vmin=-550.568816, vmax=2479.488551
✔ charts: COMA-DefaultReward | x=timesteps_total | y=episode_reward_mean | norm vmin=-550.568816 vmax=2479.488551
✔ charts: COMA-RewardHard | x=timesteps_total | y=episode_reward_mean | norm vmin=-550.568816 vmax=2479.488551
✔ charts: COMA-RewardLowerQuantile | x=timesteps_total | y=episode_reward_mean | norm vmin=-550.568816 vmax=2479.488551
✔ charts: COMA-RewardMaxMedian | x=timesteps_total | y=episode_reward_mean | norm vmin=-550.568816 vmax=2479.488551
✔ charts: HAPPO-DefaultReward | x=timesteps_total | y=episode_reward_mean | norm vmin=-550.568816 vmax=2479.488551
✔ charts: HAPPO-RewardHard | x=timesteps_total | y=episode_reward_mean | norm vmin=-550.568816 vmax=2479.488551
✔ charts: HAPPO-RewardLowerQuantile | x=timesteps_total | y=episode_reward_mean | norm vmin=-550.568816 vmax=2479.488551
✔ charts: HAPP

'/mnt/ssd1/wesley/BusEnv/runs_abril_results'